In [ ]:
# ──────────────────────────────────────────────────────────────
# main.ipynb: 사용자의 요청을 적절한 서비스 서버로 전달
# 통합 게이트웨이 역할- 브라우저와 유일하게 통신
# port 3000
# ──────────────────────────────────────────────────────────────

In [1]:
# ──────────────────────────────────────────────────────────────
# FE와 직접 통신하는 곳이므로 CORS 설정 진행
# 이때 analyze/ ocr 서버와 통신하는 것은 내부 서버 대 서버 통신이므로 추가 cors 설정 필요x
# ──────────────────────────────────────────────────────────────
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from database import save_analysis_result

import os
import httpx
import uvicorn
import threading


load_dotenv()

app = FastAPI(title="Main Gateway Server")

app.add_middleware(
    CORSMiddleware,
    # allow_origins=["*"],
    allow_origins=["http://127.0.0.1:5500", "http://localhost:5500"], # 실제 주소를 명시
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [2]:
# 내부 서버 주소 설정
ANALYZE_URL = os.getenv("ANALYZE_SERVER_URL", "http://localhost:8000")
OCR_URL = os.getenv("OCR_SERVER_URL", "http://localhost:8001")

In [3]:
def checkAllowedTypes(image) -> bool:
    allowedTypes = ['image/jpeg', 'image/png', 'image/webp', 'image/gif']
    if image.content_type not in allowedTypes:
        return False;
        
    return True;


In [4]:
@app.post("/api/analyze")
async def process_all(
    image: UploadFile = File(...),
    question: str = Form(None)
):
    """ 탭 1용: 이미지 분석 및 질문 답변 전송 """
    if not checkAllowedTypes(image):
        raise HTTPException(status_code=400, detail='지원하지 않는 이미지 형식입니다.')
    
    image_bytes = await image.read()

    async with httpx.AsyncClient(timeout=60.0) as client:
        response = await client.post(
            f"{ANALYZE_URL}/analyze", 
            files={"image": (image.filename, image_bytes, image.content_type)},
            data={"question": question}
        )

    # 2. DB에 저장 (추가된 부분)
    # 파일 원본 이름을 저장합니다.
    tmp = response.json()
    
    save_analysis_result(
        file_name=image.filename, 
        question=question, 
        answer=tmp.get('extracted_text'), 
        used_model=tmp.get('model')
    )

    return response.json()

In [ ]:
@app.post("/api/ocr")
async def process_all(
    image: UploadFile = File(...),
):
    """ 탭 2용: 이미지 OCR 출력 """
    if not checkAllowedTypes(image):
        raise HTTPException(status_code=400, detail='지원하지 않는 이미지 형식입니다.')
        
    image_bytes = await image.read()
    
    async with httpx.AsyncClient(timeout=60.0) as client:
        response = await client.post(
            f"{OCR_URL}/ocr",
            files={"image": (image.filename, image_bytes, image.content_type)}
        )

    
    # 2. DB에 저장 (추가된 부분)
    # 파일 원본 이름을 저장합니다.
    tmp = response.json()
    
    save_analysis_result(
        file_name=image.filename, 
        answer=tmp.get('extracted_text'), 
        used_model=tmp.get('model')
    )
            
    return response.json()

In [6]:
def run_server():
    # .env에서 포트를 가져오거나 기본값 8001 사용
    port = int(os.getenv("MAIN_PORT", 3000))

    # 노트북 충돌 방지를 위해 uvicorn.run 사용
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

# 이미 서버가 돌고 있는지 확인 (재실행 시 에러 방지용)
# 주피터에서 셀을 여러 번 누를 때를 대비해 thread가 살아있는지 체크하면 좋습니다.
ocr_thread = threading.Thread(target=run_server, daemon=True)
ocr_thread.start()

print(f"🚀 메인 게이트웨이 서버가 백그라운드에서 시작되었습니다!")
print(f"문서 주소: http://localhost:{os.getenv('MAIN_PORT', 3000)}/docs")

🚀 메인 게이트웨이 서버가 백그라운드에서 시작되었습니다!
문서 주소: http://localhost:3000/docs


INFO:     Started server process [16364]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:3000 (Press CTRL+C to quit)


✅ DB 저장 완료: 스크린샷 2026-04-23 155127.png
INFO:     127.0.0.1:62741 - "POST /api/analyze HTTP/1.1" 200 OK
